# P05-P09 · Horarios libres y cobertura de carga nocturna

> **Donde hay buses entre las 00:00 y las 05:00, ¿hay dónde cargar la tarjeta?**

GTFS Red Movilidad y horarios publicados en los cinco Excel de lugares bip!.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/carga_nocturna/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# Preparación: clona el repositorio y descarga los datos públicos.
# Los datos NO vienen del repositorio: se descargan del portal de datos
# abiertos, porque el laboratorio no redistribuye datos de terceros.
!git clone --depth 1 https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git lab-dtpm
%cd lab-dtpm
!pip install --quiet 'pandas>=2.2' 'numpy>=1.26' openpyxl XlsxWriter matplotlib
!python scripts/fetch_data.py gtfs-red-movilidad bip-lugares-carga


## 1 · El problema

La cobertura nocturna une tres preguntas: dónde pasa un bus entre las 00:00 y las
05:00, si hay un local de carga a 300 metros y si ese local está abierto. Las dos
primeras se calculan; la tercera tropieza con un dato ausente: 1 579 de los 1 927
locales no publican horario.

El GTFS agrega otra trampa. En expediciones por frecuencia, `arrival_time` es un
desplazamiento desde el inicio y no una hora de reloj. Filtrarlo como texto inventa
1 681 paradas nocturnas. La cifra falsa sobrestima solo 15,98 %, suficiente para parecer
plausible. La respuesta correcta no es un sí/no: tiene una tercera categoría,
indeterminada, y se expresa como intervalo.


## 2 · La solución que se va a aplicar

Se auditan las ventanas de servicio, se mide el error del filtro ingenuo y se obtiene
la intensidad nocturna sin expandir millones de filas. En paralelo se interpreta cada
texto de horario y se cuantifica la cobertura del campo. El cruce espacial clasifica
cada parada en cuatro estados observables y el informe final transforma la falta de
datos en un intervalo y en una lista concreta de información que falta recopilar.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.carga_nocturna import solution as p09

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Auditar las ventanas de servicio


Se mide qué servicios y ventanas aportan llegadas a la madrugada.

> ⚠️ Una ventana que empieza antes de medianoche puede aportar servicio después de las 00:00.


In [ ]:
resultados = p09.analizar()
resultados['ventanas_por_servicio']


### Paso 2 · Demostrar por qué el filtro textual falla


El atajo se calcula al lado del método correcto y se mide su error.

> ⚠️ El 100 % de las filas que toma el atajo son desplazamientos relativos, no horas.


In [ ]:
resultados['atajo_comparado']


### Paso 3 · Describir la intensidad nocturna


No basta con saber que una parada aparece: se cuentan llegadas y se muestran percentiles.

**En Excel:** `=PERCENTIL.INC(rango_llegadas;0,5)`

Debe dar: **23 llegadas medianas en cinco horas**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Una sola llegada y 459 llegadas no representan la misma disponibilidad.


In [ ]:
resultados['intensidad']


### Paso 4 · Convertir horarios libres en datos


Cada texto se interpreta en días y tramos, y los no interpretables permanecen visibles.

> ⚠️ Vacío no significa cerrado: significa que el publicador no entregó el campo.


In [ ]:
resultados['horarios_interpretados']


### Paso 5 · Medir la cobertura del campo horario


La ausencia se reporta por tipo de local, no como una nota al pie.

**En Excel:** `=CONTAR.SI.CONJUNTO(tipo;A2;horario;"<>")/CONTAR.SI(tipo;A2)`

Debe dar: **18,1 % del total con horario**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Promediar por archivo daría 80 %: cuatro fuentes completas y una vacía, sin ponderar sus tamaños.


In [ ]:
resultados['cobertura_del_campo']


### Paso 6 · Clasificar en cuatro estados observables


La respuesta conserva cubierta, sin local, cerrado verificado e indeterminado.

> ⚠️ Reducir a cubierto/no cubierto mezcla 5 536 incógnitas con evidencia negativa.


In [ ]:
resultados['respuesta']


### Paso 7 · Comprobar estabilidad por tipo de día


Laboral, sábado y domingo se publican separados para no ocultar variación.

> ⚠️ Sumar los servicios repite una parada hasta tres veces; cada denominador se interpreta por separado.


In [ ]:
resultados['por_servicio']


### Paso 8 · Expresar la respuesta como intervalo


El mínimo usa solo evidencia; el máximo muestra el caso favorable de todos los desconocidos.

**En Excel:** `=(abiertos_verificados+indeterminados)/total_nocturnas`

Debe dar: **0,00 % a 52,62 %**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Elegir el punto medio no agrega información: inventa una probabilidad que los datos no contienen.


In [ ]:
resultados['intervalo']


### Paso 9 · Convertir la incertidumbre en una tarea


Se identifica qué fuente completar y cuántas paradas desbloquea.

> ⚠️ Pedir 'mejorar los datos' sin señalar el campo y el impacto no es una recomendación ejecutable.


In [ ]:
resultados['lo_que_falta']


## 4 · El resultado

Hay 10 520 paradas con servicio nocturno. La cobertura abierta verificable es 0 %, pero 52,62 % permanece indeterminado: el dato correcto es un intervalo, no un cero definitivo.

## 5 · Qué queda aprendido

1. Horario relativo y hora absoluta son magnitudes distintas aunque compartan formato.
2. Vacío, cero y falso no son intercambiables.
3. Una clasificación honesta necesita una tercera respuesta cuando falta observación.
4. El ancho del intervalo mide el valor potencial de completar el dato.
